In [1]:
from __future__ import annotations

import sys
from copy import deepcopy
from pathlib import Path

import torch
import yaml
from torch.utils.data import DataLoader

In [2]:
# ── CONFIG — change this to switch datasets ───────────────────────────────────
CONFIG_FILE = "config_cifar100.yaml"          # options: config.yaml | config_cifar100.yaml | config_domainnet_isolated.yaml
# ─────────────────────────────────────────────────────────────────────────────

def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "configs" / "config.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find configs/config.yaml")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT.parent))

from ebm_unlearning.src.data.dataset import DatasetSpec, load_dataset
from ebm_unlearning.src.data.domainnet import DomainNetSubset
from ebm_unlearning.src.data.split import ForgetSpec, RetainSpec, split_forget_retain, train_holdout_split
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import load_pretrained
from ebm_unlearning.src.training.unlearn import UnlearnConfig, unlearn
from ebm_unlearning.src.utils.logging import setup_logger
from ebm_unlearning.src.utils.seed import set_seed
from ebm_unlearning.src.utils.tracking import make_tracker


In [3]:
with open(ROOT / "configs" / CONFIG_FILE, "r") as f:
    cfg = yaml.safe_load(f)

set_seed(int(cfg["seed"]))

from datetime import datetime

device = torch.device(cfg.get("device", "cpu"))
logger = setup_logger("unlearn", log_file=str(ROOT / "outputs" / "logs" / "unlearn.log"))
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
tracker = make_tracker(
    "tensorboard",
    log_dir=str(ROOT / "outputs" / "tensorboard" / cfg["data"]["dataset"] / "unlearn" / run_id),
)

dataset_name = cfg["data"]["dataset"]
forget_label = int(cfg["data"]["forget"]["class_label"])
forget_mode   = cfg["data"]["forget"]["mode"]

# ── Load dataset (DomainNet uses DomainNetSubset, others use load_dataset) ────
if dataset_name == "domainnet":
    dset = DomainNetSubset(
        root=str(ROOT / cfg["data"]["data_dir"]),
        classes=cfg["data"]["classes"],
        domains=cfg["data"]["domains"],
    )
    class_name    = cfg["data"]["forget"].get("class_name", cfg["data"]["classes"][forget_label])
    forget_domain = cfg["data"]["forget"].get("domain", "")
    checkpoint_name = f"domainnet_unlearned_{class_name}_{forget_domain}.pt"
    forget_spec = ForgetSpec(mode=forget_mode, class_label=forget_label, domain=forget_domain)
else:
    spec = DatasetSpec(name=dataset_name, data_dir=str(ROOT / cfg["data"]["data_dir"]), train=True, download=True)
    dset = load_dataset(spec)
    class_name = dset.classes[forget_label] if hasattr(dset, "classes") else str(forget_label)
    checkpoint_name = f"{dataset_name}_unlearned_{class_name}.pt"
    forget_spec = ForgetSpec(mode=forget_mode, class_label=forget_label)

checkpoint_path = str(ROOT / "outputs" / "checkpoints" / checkpoint_name)
print(f"Dataset  : {dataset_name}")
print(f"Forget   : {class_name} (label={forget_label})")
print(f"Checkpoint will be saved to: {checkpoint_path}")

retain_spec = RetainSpec(mode=cfg["data"]["retain"]["mode"])
forget_all, retain_all = split_forget_retain(dset, forget_spec, retain_spec)

holdout_fraction = float(cfg["evaluation"]["holdout_fraction"])
forget_train, forget_holdout = train_holdout_split(forget_all, holdout_fraction, seed=int(cfg["seed"]))
retain_train, retain_holdout = train_holdout_split(retain_all, holdout_fraction, seed=int(cfg["seed"]) + 1)

batch_size  = int(cfg["data"]["batch_size"])
num_workers = int(cfg["data"]["num_workers"])
forget_loader         = DataLoader(forget_train,   batch_size=batch_size, shuffle=True,  num_workers=0, drop_last=True)
retain_loader         = DataLoader(retain_train,   batch_size=batch_size, shuffle=True,  num_workers=0, drop_last=True)
forget_holdout_loader = DataLoader(forget_holdout, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=True)
retain_holdout_loader = DataLoader(retain_holdout, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=True)

print(f"Forget train={len(forget_train)}  holdout={len(forget_holdout)}")
print(f"Retain train={len(retain_train)}  holdout={len(retain_holdout)}")


2026-06-05 01:19:33.052727: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-05 01:19:33.128580: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-05 01:19:34.981570: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[data] loading cifar100 (train=True, download=True) from /home/owais/machine unlearning/ebm_unlearning/data
Dataset  : cifar100
Forget   : rocket (label=69)
Checkpoint will be saved to: /home/owais/machine unlearning/ebm_unlearning/outputs/checkpoints/cifar100_unlearned_rocket.pt
Forget train=400  holdout=100
Retain train=39600  holdout=9900


In [4]:
E0 = EnergyModel(
    in_channels=int(cfg["model"]["in_channels"]),
    hidden_dim=int(cfg["model"]["hidden_dim"]),
    num_classes=int(cfg["model"].get("num_classes", 10)),
    embed_dim=int(cfg["model"].get("embed_dim", 128)),
    backbone=str(cfg["model"].get("backbone", "conv")),
    finetune_stages=int(cfg["model"].get("finetune_stages", 1)),
    imagenet_pretrained=bool(cfg["model"].get("imagenet_pretrained", True)),
)
E0 = load_pretrained(E0, str(ROOT / cfg["pretrain"]["checkpoint_path"]), device=device)

E = deepcopy(E0)

un_cfg = UnlearnConfig(
    steps=int(cfg["unlearning"]["steps"]),
    lr=float(cfg["unlearning"]["lr"]),
    weight_decay=float(cfg["unlearning"]["weight_decay"]),
    lambda_f=float(cfg["unlearning"]["lambda_f"]),
    lambda_r=float(cfg["unlearning"]["lambda_r"]),
    lambda_m=float(cfg["unlearning"]["lambda_m"]),
    lambda_e=float(cfg["unlearning"]["lambda_e"]),
    margin=float(cfg["unlearning"]["margin"]),
    neg_k_forget=int(cfg["unlearning"].get("neg_k_forget", 0)),
    log_every=int(cfg["unlearning"]["log_every"]),
    checkpoint_path=checkpoint_path,   # dynamic: {dataset}_unlearned_{classname}.pt
)


In [5]:
E = unlearn(
    E,
    E0,
    forget_loader,
    retain_loader,
    device=device,
    cfg=un_cfg,
    logger=logger,
    tracker=tracker,
    seed=int(cfg["seed"]),
    forget_holdout_loader=forget_holdout_loader,
    retain_holdout_loader=retain_holdout_loader,
)
tracker.close()
print(f"Saved to: {checkpoint_path}")

[2026-06-05 01:19:38,497] [INFO] [unlearn] step=0 margin=5.0000 gap_fw=-0.5831 total=44.123268 forget=6.976139 retain=3.398968 margin_loss=3.153338 energy_reg=4.108860
[2026-06-05 01:19:45,389] [INFO] [unlearn] step=50 margin=5.0000 gap_fw=1.5700 total=17.642092 forget=5.230460 retain=1.050425 margin_loss=1.900803 energy_reg=6.574203
[2026-06-05 01:19:51,444] [INFO] [unlearn] step=100 margin=5.0000 gap_fw=2.8058 total=20.758602 forget=3.694963 retain=1.633825 margin_loss=0.710058 energy_reg=15.332502
[2026-06-05 01:19:57,472] [INFO] [unlearn] step=150 margin=5.0000 gap_fw=5.0657 total=15.848796 forget=2.935081 retain=1.268466 margin_loss=0.207680 energy_reg=21.375557
[2026-06-05 01:20:02,651] [INFO] [unlearn] step=200 margin=5.0000 gap_fw=6.4580 total=20.472084 forget=2.232136 retain=1.809614 margin_loss=0.111971 energy_reg=31.840540
[2026-06-05 01:20:08,799] [INFO] [unlearn] step=250 margin=5.0000 gap_fw=6.8599 total=12.948008 forget=1.591958 retain=1.117932 margin_loss=0.132977 energ

Saved to: /home/owais/machine unlearning/ebm_unlearning/outputs/checkpoints/cifar100_unlearned_rocket.pt


In [ ]:
# ── Stage 2: Repair retain accuracy ──────────────────────────────────────────
# Run supervised energy contrast loss on retain data only.
# Backbone relearns retain discrimination; label_emb[forget_class] gets
# zero gradient (no forget-class images) so forgetting is preserved.
from ebm_unlearning.src.losses.pretrain import supervised_energy_contrast_loss
import torch

REPAIR_STEPS  = 300
REPAIR_LR     = 1e-5
REPAIR_K_NEG  = 5
REPAIR_MARGIN = 1.0

E.train()
opt_repair = torch.optim.Adam(E.parameters(), lr=REPAIR_LR)
g_repair   = torch.Generator(device="cpu").manual_seed(int(cfg["seed"]) + 99)

def _cycle(loader):
    while True:
        for batch in loader:
            yield batch

num_classes_repair = int(cfg["model"].get("num_classes", 10))
r_iter = _cycle(retain_loader)

print(f"Repair: {REPAIR_STEPS} steps  lr={REPAIR_LR}  k_neg={REPAIR_K_NEG}")
for step in range(REPAIR_STEPS):
    xr, yr = next(r_iter)[:2]
    xr = xr.to(device); yr = yr.to(device).long()
    b  = xr.shape[0]
    neg_rows = []
    for _ in range(REPAIR_K_NEG):
        r = torch.randint(0, num_classes_repair - 1, (b,), generator=g_repair)
        neg_rows.append(r + (r >= yr.cpu()).long())
    y_neg = torch.stack(neg_rows, dim=1).to(device)
    loss = supervised_energy_contrast_loss(E, xr, yr, y_neg, REPAIR_MARGIN)
    opt_repair.zero_grad(set_to_none=True)
    loss.backward()
    opt_repair.step()
    if step % 50 == 0:
        print(f"  [repair] step={step}  loss={loss.item():.6f}")

import torch as _t
_t.save({"model": E.state_dict()}, checkpoint_path)
print(f"Repaired checkpoint saved → {checkpoint_path}")


In [6]:
from __future__ import annotations
# ══════════════════════════════════════════════════════════════════════════════
#  EVALUATION — Classification Accuracy + MIA
#  Set these two variables and run. No other cells needed.
# ══════════════════════════════════════════════════════════════════════════════
CONFIG_FILE          = "config_cifar100.yaml"   # config.yaml | config_cifar100.yaml | config_domainnet_isolated.yaml
UNLEARNED_CHECKPOINT = "outputs/checkpoints/cifar100_unlearned_rocket.pt"
# ══════════════════════════════════════════════════════════════════════════════
import sys
import numpy as np
from pathlib import Path
import torch
import yaml
from torch.utils.data import DataLoader

def _find_root():
    p = Path(".").resolve()
    for _ in range(6):
        if (p / "configs" / "config.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("project root not found")

ROOT = _find_root()
sys.path.insert(0, str(ROOT.parent))

from ebm_unlearning.src.data.dataset import DatasetSpec, load_dataset
from ebm_unlearning.src.data.domainnet import DomainNetSubset
from ebm_unlearning.src.data.split import ForgetSpec, RetainSpec, split_forget_retain, train_holdout_split
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import load_pretrained
from ebm_unlearning.src.evaluation.classification import predict_argmin_energy
from ebm_unlearning.src.evaluation.metrics import collect_energies, membership_inference_proxy

with open(ROOT / "configs" / CONFIG_FILE) as f:
    cfg = yaml.safe_load(f)

device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_name = cfg["data"]["dataset"]
forget_label = int(cfg["data"]["forget"]["class_label"])
forget_mode  = cfg["data"]["forget"]["mode"]
num_classes  = int(cfg["model"].get("num_classes", 10))
y_chunk      = max(5, num_classes // 20)

def _make_model():
    return EnergyModel(
        in_channels=int(cfg["model"]["in_channels"]),
        hidden_dim=int(cfg["model"]["hidden_dim"]),
        num_classes=num_classes,
        embed_dim=int(cfg["model"].get("embed_dim", 128)),
        backbone=str(cfg["model"].get("backbone", "conv")),
        finetune_stages=int(cfg["model"].get("finetune_stages", 1)),
        imagenet_pretrained=bool(cfg["model"].get("imagenet_pretrained", True)),
    )

print("Loading models...")
E0 = load_pretrained(_make_model(), str(ROOT / cfg["pretrain"]["checkpoint_path"]), device=device)
E  = load_pretrained(_make_model(), str(ROOT / UNLEARNED_CHECKPOINT), device=device)
E0.eval(); E.eval()
print(f"  Pretrained : {cfg['pretrain']['checkpoint_path']}")
print(f"  Unlearned  : {UNLEARNED_CHECKPOINT}")

if dataset_name == "domainnet":
    dset_train = DomainNetSubset(root=str(ROOT / cfg["data"]["data_dir"]),
                                 classes=cfg["data"]["classes"], domains=cfg["data"]["domains"])
    forget_domain = cfg["data"]["forget"].get("domain", "")
    class_name    = cfg["data"]["forget"].get("class_name", cfg["data"]["classes"][forget_label])
    forget_spec   = ForgetSpec(mode=forget_mode, class_label=forget_label, domain=forget_domain)
    dset_test = dset_train   # DomainNet has no separate test split
else:
    dset_train    = load_dataset(DatasetSpec(name=dataset_name, data_dir=str(ROOT / cfg["data"]["data_dir"]), train=True,  download=True))
    dset_test     = load_dataset(DatasetSpec(name=dataset_name, data_dir=str(ROOT / cfg["data"]["data_dir"]), train=False, download=True))
    class_name    = dset_train.classes[forget_label] if hasattr(dset_train, "classes") else str(forget_label)
    forget_spec   = ForgetSpec(mode=forget_mode, class_label=forget_label)
    forget_domain = None

# MIA uses training splits (train vs holdout) ─────────────────────────────────
forget_all, retain_all = split_forget_retain(dset_train, forget_spec, RetainSpec())
hf   = float(cfg["evaluation"]["holdout_fraction"])
seed = int(cfg["seed"])
forget_train, forget_holdout = train_holdout_split(forget_all, hf, seed=seed)
retain_train, retain_holdout = train_holdout_split(retain_all, hf, seed=seed + 1)
bs = int(cfg["data"]["batch_size"])
fl_tr = DataLoader(forget_train,   batch_size=bs, shuffle=False, num_workers=0)
fl_ho = DataLoader(forget_holdout, batch_size=bs, shuffle=False, num_workers=0)
rl_tr = DataLoader(retain_train,   batch_size=bs, shuffle=False, num_workers=0)
rl_ho = DataLoader(retain_holdout, batch_size=bs, shuffle=False, num_workers=0)

# Classification uses test set ─────────────────────────────────────────────────
forget_test, retain_test = split_forget_retain(dset_test, forget_spec, RetainSpec())
fl_test = DataLoader(forget_test, batch_size=bs, shuffle=False, num_workers=0)
rl_test = DataLoader(retain_test, batch_size=bs, shuffle=False, num_workers=0)
print(f"  Test  forget={len(forget_test)}  retain={len(retain_test)}")
print(f"  Train forget={len(forget_train)} holdout={len(forget_holdout)} (MIA only)")

def _clf(model, loader):
    return predict_argmin_energy(model, loader, device=device, num_classes=num_classes, y_chunk=y_chunk)
def _acc(yt, yp):
    return float(np.mean(yt == yp)) if len(yt) else float("nan")

print("Classifying on test set...")
yt_f, yp_f_pre = _clf(E0, fl_test);  _,    yp_f_unl = _clf(E,  fl_test)
yt_r, yp_r_pre = _clf(E0, rl_test);  _,    yp_r_unl = _clf(E,  rl_test)

fa_pre = _acc(yt_f, yp_f_pre);  fa_unl = _acc(yt_f, yp_f_unl)
ra_pre = _acc(yt_r, yp_r_pre);  ra_unl = _acc(yt_r, yp_r_unl)
fr = (fa_pre - fa_unl) / fa_pre if fa_pre > 0 else float("nan")
mu = ra_unl / ra_pre             if ra_pre > 0 else float("nan")

print("Computing MIA on training splits...")
mia_pre = membership_inference_proxy(collect_energies(E0, fl_tr, device=device), collect_energies(E0, fl_ho, device=device))
mia_unl = membership_inference_proxy(collect_energies(E,  fl_tr, device=device), collect_energies(E,  fl_ho, device=device))
mia_ret = membership_inference_proxy(collect_energies(E,  rl_tr, device=device), collect_energies(E,  rl_ho, device=device))

W = 54
print()
print("=" * W)
print(f"  RESULTS — {dataset_name.upper()} — forget: {class_name}")
print("=" * W)
print(f"  {'Metric':<36} {'Pretrained':>9} {'Unlearned':>9}")
print(f"  {'-' * (W - 2)}")
print(f"  {'Forget accuracy':<36} {fa_pre:>8.1%} {fa_unl:>9.1%}")
print(f"  {'Retain accuracy':<36} {ra_pre:>8.1%} {ra_unl:>9.1%}")
print(f"  {'Forgetting rate  (higher is better)':<36} {'—':>9} {fr:>9.1%}")
print(f"  {'Model utility    (higher is better)':<36} {'—':>9} {mu:>9.1%}")
print(f"  {'-' * (W - 2)}")
print(f"  {'MIA forget pretrained  (higher=overfit)':<36} {mia_pre:>9.4f} {'—':>9}")
print(f"  {'MIA forget unlearned   (0.5=perfect)':<36} {'—':>9} {mia_unl:>9.4f}")
print(f"  {'MIA retain unlearned   (0.5=perfect)':<36} {'—':>9} {mia_ret:>9.4f}")
print("=" * W)

if dataset_name == "domainnet":
    from ebm_unlearning.src.data.domainnet import DOMAINS
    from ebm_unlearning.src.data.dataset import IndexedSubset
    print(f"\n  Cross-domain breakdown — class: {class_name}")
    print(f"  {'Domain':<12} {'Pretrained':>11} {'Unlearned':>11} {'Drop':>8}")
    print(f"  {'-' * 46}")
    for dom in DOMAINS:
        d    = DomainNetSubset(root=str(ROOT / cfg["data"]["data_dir"]),
                               classes=cfg["data"]["classes"], domains=[dom])
        mask = d.targets == forget_label
        if mask.sum() == 0:
            continue
        idx  = torch.nonzero(mask, as_tuple=False).squeeze(1)
        dl   = DataLoader(__import__('ebm_unlearning.src.data.dataset', fromlist=['IndexedSubset']).IndexedSubset(d, idx),
                          batch_size=32, shuffle=False, num_workers=0)
        _, yp0 = _clf(E0, dl)
        _, yp1 = _clf(E,  dl)
        yt_d   = d.targets[idx].numpy()
        a0 = _acc(yt_d, yp0);  a1 = _acc(yt_d, yp1)
        tag = " <- forget" if dom == forget_domain else ""
        print(f"  {dom:<12} {a0:>10.1%} {a1:>11.1%} {a0-a1:>+8.1%}{tag}")


Loading models...
  Pretrained : outputs/checkpoints/ebm_pretrained_cifar100.pt
  Unlearned  : outputs/checkpoints/cifar100_unlearned_rocket.pt
[data] loading cifar100 (train=True, download=True) from /home/owais/machine unlearning/ebm_unlearning/data
[data] loading cifar100 (train=False, download=True) from /home/owais/machine unlearning/ebm_unlearning/data
  Test  forget=100  retain=9900
  Train forget=400 holdout=100 (MIA only)
Classifying on test set...
Computing MIA on training splits...

  RESULTS — CIFAR100 — forget: rocket
  Metric                               Pretrained Unlearned
  ----------------------------------------------------
  Forget accuracy                         74.0%      0.0%
  Retain accuracy                         45.9%     36.5%
  Forgetting rate  (higher is better)          —    100.0%
  Model utility    (higher is better)          —     79.5%
  ----------------------------------------------------
  MIA forget pretrained  (higher=overfit)    0.4998        

In [ ]:
from __future__ import annotations
# ══════════════════════════════════════════════════════════════════════════════
#  EVALUATION — Per-Class Accuracy Table
#  Set these two variables and run. No other cells needed.
# ══════════════════════════════════════════════════════════════════════════════
CONFIG_FILE          = "config_cifar100.yaml"   # config.yaml | config_cifar100.yaml | config_domainnet_isolated.yaml
UNLEARNED_CHECKPOINT = "outputs/checkpoints/cifar100_unlearned_rocket.pt"
# ══════════════════════════════════════════════════════════════════════════════

import sys
import numpy as np
from pathlib import Path
import torch
import yaml
from torch.utils.data import DataLoader

def _find_root():
    p = Path(".").resolve()
    for _ in range(6):
        if (p / "configs" / "config.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("project root not found")

ROOT = _find_root()
sys.path.insert(0, str(ROOT.parent))

from ebm_unlearning.src.data.dataset import DatasetSpec, load_dataset
from ebm_unlearning.src.data.domainnet import DomainNetSubset
from ebm_unlearning.src.data.split import ForgetSpec, RetainSpec, split_forget_retain, train_holdout_split
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import load_pretrained
from ebm_unlearning.src.evaluation.classification import predict_argmin_energy

with open(ROOT / "configs" / CONFIG_FILE) as f:
    cfg = yaml.safe_load(f)

device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_name = cfg["data"]["dataset"]
forget_label = int(cfg["data"]["forget"]["class_label"])
num_classes  = int(cfg["model"].get("num_classes", 10))
y_chunk      = max(5, num_classes // 20)

def _make_model():
    return EnergyModel(
        in_channels=int(cfg["model"]["in_channels"]),
        hidden_dim=int(cfg["model"]["hidden_dim"]),
        num_classes=num_classes,
        embed_dim=int(cfg["model"].get("embed_dim", 128)),
        backbone=str(cfg["model"].get("backbone", "conv")),
        finetune_stages=int(cfg["model"].get("finetune_stages", 1)),
        imagenet_pretrained=bool(cfg["model"].get("imagenet_pretrained", True)),
    )

print("Loading models...")
E0 = load_pretrained(_make_model(), str(ROOT / cfg["pretrain"]["checkpoint_path"]), device=device)
E  = load_pretrained(_make_model(), str(ROOT / UNLEARNED_CHECKPOINT), device=device)
E0.eval(); E.eval()

# ── Load evaluation dataset ───────────────────────────────────────────────────
if dataset_name == "domainnet":
    dset = DomainNetSubset(
        root=str(ROOT / cfg["data"]["data_dir"]),
        classes=cfg["data"]["classes"],
        domains=cfg["data"]["domains"],
    )
    class_names   = dset.classes
    forget_domain = cfg["data"]["forget"].get("domain", "")
    class_name    = cfg["data"]["forget"].get("class_name", class_names[forget_label])
else:
    # Use test split for clean per-class evaluation
    spec = DatasetSpec(name=dataset_name, data_dir=str(ROOT / cfg["data"]["data_dir"]),
                       train=False, download=True)
    dset = load_dataset(spec)
    class_names = dset.classes if hasattr(dset, "classes") else [str(i) for i in range(num_classes)]
    class_name  = class_names[forget_label]
    forget_domain = None

loader = DataLoader(dset, batch_size=int(cfg["data"]["batch_size"]),
                    shuffle=False, num_workers=0)
print(f"Evaluating on {len(dset)} {'test' if dataset_name != 'domainnet' else 'full'} samples...")

def _clf(model):
    return predict_argmin_energy(model, loader, device=device,
                                 num_classes=num_classes, y_chunk=y_chunk)

print("Classifying with pretrained model...")
yt, yp_pre = _clf(E0)
print("Classifying with unlearned model...")
_,  yp_unl = _clf(E)

# ── Per-class accuracy ────────────────────────────────────────────────────────
def _per_class_acc(yt, yp):
    return {c: float(np.mean(yp[yt == c] == c)) if (yt == c).sum() > 0 else float("nan")
            for c in range(num_classes)}

acc_pre = _per_class_acc(yt, yp_pre)
acc_unl = _per_class_acc(yt, yp_unl)

# ── Classes to show in table ──────────────────────────────────────────────────
# Empty = show all (good for analysis). Set labels to restrict for paper tables.
# Example for rocket on CIFAR-100:
#   forget class + vehicles_2 superclass + vehicles_1 + unrelated controls
FOCUS_CLASSES = []   # e.g. [69, 41, 78, 82, 86, 8, 13, 48, 90, 51, 54, 0, 22]
# ─────────────────────────────────────────────────────────────────────────────

show = set(FOCUS_CLASSES) if FOCUS_CLASSES else set(range(num_classes))
show.add(forget_label)   # always include the forget class

W = 62
print()
print("=" * W)
print(f"  PER-CLASS ACCURACY — {dataset_name.upper()} — forget: {class_name}")
if FOCUS_CLASSES:
    print(f"  (showing {len(show)} selected classes)")
print("=" * W)
print(f"  {'Class':<22} {'Label':>5} {'Pretrained':>11} {'Unlearned':>10} {'Change':>8}  ")
print(f"  {'-' * (W - 2)}")

for c in range(num_classes):
    if c not in show:
        continue
    name = class_names[c] if c < len(class_names) else str(c)
    pre  = acc_pre[c]
    unl  = acc_unl[c]
    chg  = unl - pre
    flag = " <- FORGET" if c == forget_label else ""
    chg_str = f"{chg:>+7.1%}"
    print(f"  {name:<22} {c:>5} {pre:>10.1%} {unl:>10.1%} {chg_str}{flag}")

print(f"  {'-' * (W - 2)}")
overall_pre = float(np.mean(yt == yp_pre))
overall_unl = float(np.mean(yt == yp_unl))
print(f"  {'OVERALL':<22} {'':>5} {overall_pre:>10.1%} {overall_unl:>10.1%} {overall_unl - overall_pre:>+7.1%}")
print("=" * W)


Loading models...
[data] loading cifar100 (train=False, download=True) from /home/owais/machine unlearning/ebm_unlearning/data
Evaluating on 10000 test samples...
Classifying with pretrained model...
Classifying with unlearned model...


In [ ]:
from __future__ import annotations
# ══════════════════════════════════════════════════════════════════════════════
#  EVALUATION — EBM Energy Metrics
#  Set these two variables and run. No other cells needed.
# ══════════════════════════════════════════════════════════════════════════════
CONFIG_FILE          = "config.yaml"   # config.yaml | config_cifar100.yaml | config_domainnet_isolated.yaml
UNLEARNED_CHECKPOINT = "outputs/checkpoints/cifar10_unlearned_airplane.pt"
# ══════════════════════════════════════════════════════════════════════════════

import sys
import numpy as np
from pathlib import Path
import torch
import yaml
from torch.utils.data import DataLoader

def _find_root():
    p = Path(".").resolve()
    for _ in range(6):
        if (p / "configs" / "config.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("project root not found")

ROOT = _find_root()
sys.path.insert(0, str(ROOT.parent))

from ebm_unlearning.src.data.dataset import DatasetSpec, load_dataset
from ebm_unlearning.src.data.domainnet import DomainNetSubset
from ebm_unlearning.src.data.split import ForgetSpec, RetainSpec, split_forget_retain, train_holdout_split
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import load_pretrained
from ebm_unlearning.src.evaluation.metrics import collect_energies

with open(ROOT / "configs" / CONFIG_FILE) as f:
    cfg = yaml.safe_load(f)

device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_name = cfg["data"]["dataset"]
forget_label = int(cfg["data"]["forget"]["class_label"])
forget_mode  = cfg["data"]["forget"]["mode"]
num_classes  = int(cfg["model"].get("num_classes", 10))
y_chunk      = max(5, num_classes // 20)

def _make_model():
    return EnergyModel(
        in_channels=int(cfg["model"]["in_channels"]),
        hidden_dim=int(cfg["model"]["hidden_dim"]),
        num_classes=num_classes,
        embed_dim=int(cfg["model"].get("embed_dim", 128)),
        backbone=str(cfg["model"].get("backbone", "conv")),
        finetune_stages=int(cfg["model"].get("finetune_stages", 1)),
        imagenet_pretrained=bool(cfg["model"].get("imagenet_pretrained", True)),
    )

print("Loading models...")
E0 = load_pretrained(_make_model(), str(ROOT / cfg["pretrain"]["checkpoint_path"]), device=device)
E  = load_pretrained(_make_model(), str(ROOT / UNLEARNED_CHECKPOINT), device=device)
E0.eval(); E.eval()

# ── Dataset and splits (training set — needed for correct holdout alignment) ──
if dataset_name == "domainnet":
    dset = DomainNetSubset(root=str(ROOT / cfg["data"]["data_dir"]),
                           classes=cfg["data"]["classes"], domains=cfg["data"]["domains"])
    class_name  = cfg["data"]["forget"].get("class_name", cfg["data"]["classes"][forget_label])
    forget_spec = ForgetSpec(mode=forget_mode, class_label=forget_label,
                             domain=cfg["data"]["forget"].get("domain", ""))
else:
    dset        = load_dataset(DatasetSpec(name=dataset_name, data_dir=str(ROOT / cfg["data"]["data_dir"]),
                                           train=True, download=True))
    class_name  = dset.classes[forget_label] if hasattr(dset, "classes") else str(forget_label)
    forget_spec = ForgetSpec(mode=forget_mode, class_label=forget_label)

forget_all, retain_all = split_forget_retain(dset, forget_spec, RetainSpec())
hf   = float(cfg["evaluation"]["holdout_fraction"])
seed = int(cfg["seed"])
forget_train, forget_holdout = train_holdout_split(forget_all, hf, seed=seed)
retain_train, retain_holdout = train_holdout_split(retain_all, hf, seed=seed + 1)
bs = int(cfg["data"]["batch_size"])
fl_ho = DataLoader(forget_holdout, batch_size=bs, shuffle=False, num_workers=0)
rl_tr = DataLoader(retain_train,   batch_size=bs, shuffle=False, num_workers=0)
rl_ho = DataLoader(retain_holdout, batch_size=bs, shuffle=False, num_workers=0)

# ── Collect energies ──────────────────────────────────────────────────────────
print("Collecting energies...")
f_ho_e0 = collect_energies(E0, fl_ho, device=device)   # pretrained, forget holdout
f_ho_e  = collect_energies(E,  fl_ho, device=device)   # unlearned,  forget holdout
r_ho_e0 = collect_energies(E0, rl_ho, device=device)   # pretrained, retain holdout
r_ho_e  = collect_energies(E,  rl_ho, device=device)   # unlearned,  retain holdout
r_tr_e0 = collect_energies(E0, rl_tr, device=device)   # pretrained, retain train
r_tr_e  = collect_energies(E,  rl_tr, device=device)   # unlearned,  retain train

# ── 1. Energy gap ─────────────────────────────────────────────────────────────
# mean E(forget, correct_label) - mean E(retain, correct_label)
# After unlearning: forget energy should be much higher than retain energy
eg_pre = float(f_ho_e0.mean() - r_ho_e0.mean())
eg_unl = float(f_ho_e.mean()  - r_ho_e.mean())

# ── 2. Normalized retention score ─────────────────────────────────────────────
# MSE(E_unlearned(retain), E_pretrained(retain)) / mean(|E_pretrained(retain)|)
# Normalizing by the pretrained energy scale makes this dimensionless and comparable
# across datasets. 0 = no drift. 1 = drift equal in magnitude to the energy itself.
mse_raw   = float(np.mean((r_tr_e - r_tr_e0) ** 2))
scale     = float(np.mean(np.abs(r_tr_e0)))
ret_score = mse_raw / scale if scale > 0 else float("nan")

# ── 3 & 4. Energy rank and argmax rate ────────────────────────────────────────
# For each forget-holdout image: rank the correct label's energy among all C classes.
# Rank 1 = lowest energy (model predicts this class).
# Rank C = highest energy (model maximally rejects this class).
# Mean rank close to C and argmax rate close to 100% = complete energetic forgetting.
@torch.no_grad()
def _all_class_energies(model, loader):
    rows = []
    for xb, _ in loader:
        xb = xb.to(device)
        b  = xb.shape[0]
        chunks = []
        for y0 in range(0, num_classes, y_chunk):
            ys = torch.arange(y0, min(y0 + y_chunk, num_classes), device=device, dtype=torch.long)
            xr = xb.unsqueeze(1).expand(b, ys.numel(), *xb.shape[1:]).reshape(b * ys.numel(), *xb.shape[1:])
            yr = ys.unsqueeze(0).expand(b, -1).reshape(b * ys.numel())
            chunks.append(model(xr, yr).reshape(b, ys.numel()).cpu().numpy())
        rows.append(np.concatenate(chunks, axis=1))
    return np.concatenate(rows) if rows else np.empty((0, num_classes))

print("Computing energy ranks...")
all_e0 = _all_class_energies(E0, fl_ho)
all_e  = _all_class_energies(E,  fl_ho)

def _rank_stats(all_e, lbl):
    ranks = np.array([int(np.where(np.argsort(r) == lbl)[0][0]) + 1 for r in all_e])
    return float(ranks.mean()), float(np.mean(ranks == num_classes))

rank_pre, argmax_pre = _rank_stats(all_e0, forget_label)
rank_unl, argmax_unl = _rank_stats(all_e,  forget_label)

# ── Print table ───────────────────────────────────────────────────────────────
W = 62
print()
print("=" * W)
print(f"  EBM ENERGY METRICS — {dataset_name.upper()} — forget: {class_name}")
print("=" * W)
print(f"  {'Metric':<46} {'Pre':>6} {'Unl':>6}")
print(f"  {'-' * (W - 2)}")
print(f"  {'Energy gap  E(forget) - E(retain)  [higher=better]':<46} {eg_pre:>+6.3f} {eg_unl:>+6.3f}")
print(f"  {'-' * (W - 2)}")
print(f"  {'Normalized retain drift  MSE/|E0|   [lower=better]':<46} {'—':>6} {ret_score:>6.4f}")
print(f"  {'-' * (W - 2)}")
print(f"  {'Mean energy rank of forget class  (out of ' + str(num_classes) + ')':<46} {rank_pre:>6.2f} {rank_unl:>6.2f}")
print(f"  {'Argmax rate: forget = highest energy  [higher=better]':<46} {argmax_pre:>6.1%} {argmax_unl:>6.1%}")
print("=" * W)
print()
print(f"  Interpretation:")
print(f"  Energy gap   : pretrained {eg_pre:+.3f} (forget≈retain) → unlearned {eg_unl:+.3f} (forget>>retain)")
print(f"  Retain drift : {ret_score:.4f}  ({'minimal' if ret_score < 0.1 else 'moderate' if ret_score < 0.5 else 'high'} — retain energy structure {'well preserved' if ret_score < 0.1 else 'mostly preserved' if ret_score < 0.5 else 'significantly changed'})")
print(f"  Energy rank  : {rank_pre:.2f}/10 → {rank_unl:.2f}/10  ({'complete' if rank_unl >= num_classes - 0.1 else 'partial'} energetic forgetting)")
print(f"  Argmax rate  : {argmax_pre:.1%} → {argmax_unl:.1%}  ({'complete' if argmax_unl > 0.99 else 'partial'} — forget class is {'always' if argmax_unl > 0.99 else 'sometimes'} the highest-energy label)")
